In [ ]:
!pip install bertopic
!pip install gensim
from google.colab import drive
drive.mount('/content/drive')
import sys
from bertopic import BERTopic
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/CombinedTrainingDataset.ipynb"
%run "/content/drive/MyDrive/Colab Notebooks/StopWords.ipynb"

Using Colab cache for faster access to the 'liberals-vs-conservatives-on-reddit-13000-posts' dataset.
Using Colab cache for faster access to the '1-million-reddit-comments-from-40-subreddits' dataset.
Successfully combined!
Dataset 1 rows: 12854
Dataset 2 ('politics' only) rows: 25000
Total combined rows: 37854


In [ ]:
vectorizer_model = CountVectorizer(stop_words=my_stop_words, ngram_range=(1, 2), min_df=10, token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')

In [ ]:

seed_topic_list = [
    # 1. Inflation & Cost of Living
    ["inflation", "prices", "cost", "economy", "fed", "recession", "gas", "groceries",
    "purchasing", "power", "interest", "rates", "supply", "chain", "deficit",
    "spending", "cpi", "consumer", "index", "expensive", "affordability"],
    # 2. Taxes & Wealth Distribution
    ["tax", "taxes", "irs", "wealth", "billionaires", "loophole", "income", "corporate",
    "brackets", "capital", "gains", "revenue", "billionaire", "millionaires", "fair",
    "share", "audit", "deduction", "estates", "rich", "taxation"],
    # 3. Labor Rights & Unions
    ["labor", "unions", "strike", "workers", "wages", "minimum", "pay", "benefits",
    "bargaining", "unionization", "guild", "workplace", "employment", "overtime",
    "safety", "pension", "salaries", "staffing", "contract", "exploitation"],
    # 4. Housing & Real Estate
    ["housing", "rent", "mortgage", "zoning", "homelessness", "affordable", "eviction",
    "landlord", "tenants", "property", "real", "estate", "development", "market",
    "shortage", "equity", "gentrification", "shelter", "units", "inventory"],
    # 5. Student Debt & Education
    ["debt", "student", "loans", "forgiveness", "tuition", "college", "university",
    "degree", "graduates", "interest", "borrowers", "education", "academic",
    "scholarship", "fafsa", "repayment", "cancel", "default", "higher", "learning"],
    # 6. Healthcare & Insurance
    ["insurance", "healthcare", "medicare", "hospital", "coverage", "premiums", "prescription",
    "medicaid", "deductible", "copay", "provider", "doctor", "physician", "clinic", "treatment",
    "patient", "medical", "pharma", "aca", "obamacare"],
    # 7. Abortion & Reproductive Rights
    ["abortion", "roe", "wade", "planned", "parenthood", "fetus", "reproductive", "pregnancy",
    "choice", "prolife", "prochoice", "clinic", "viability", "contraception", "birth",
    "autonomy", "rights", "women", "dobbs", "legal"],
    # 8. Pandemic & Public Health
    ["covid", "pandemic", "vaccine", "mandate", "mask", "fauci", "virus", "health",
    "quarantine", "lockdown", "outbreak", "moderna", "pfizer", "booster", "cdc",
    "transmission", "distancing", "public", "emergency", "variants"],
    # 9. Drug Policy & Decriminalization
    ["drugs", "legalization", "marijuana", "weed", "decriminalization", "addiction", "overdose",
    "cannabis", "opioid", "fentanyl", "narcotics", "treatment", "dispensary", "rehab",
    "recreational", "controlled", "substances", "prison", "sentencing", "possession"],
    # 10. Gun Control & Second Amendment
    ["gun", "guns", "nra", "amendment", "firearm", "background", "checks", "shooting",
    "violence", "weapons", "rifle", "pistol", "ban", "arms", "safety",
    "massacre", "shooter", "regulation", "carry", "concealed"],
    # 11. LGBTQ+ & Gender Issues
    ["lgbtq", "trans", "transgender", "gay", "marriage", "gender", "rights", "pride",
    "queer", "homosexuality", "bisexual", "pronouns", "identity", "nonbinary", "transition",
    "equality", "discrimination", "hormone", "bathroom", "ban"],
    # 12. Racial Justice & Civil Rights
    ["race", "racism", "systemic", "blm", "black", "white", "diversity", "equity",
    "inclusion", "prejudice", "minority", "supremacy", "segregation", "reparations", "privilege",
    "civil", "rights", "marginalized", "profiling", "justice"],
    # 13. Policing & Law Enforcement
   ["police", "cops", "brutality", "reform", "defund", "officer", "arrest", "accountability",
    "sheriff", "patrol", "shooting", "force", "misconduct", "qualified", "immunity",
    "detective", "badge", "law", "enforcement", "swat"],
    # 14. Criminal Justice & Prison Reform
    ["prison", "justice", "criminal", "reform", "incarceration", "sentence", "inmates",
    "jail", "parole", "probation", "felony", "misdemeanor", "conviction", "recidivism", "rehab",
    "solitary", "warden", "exonerated", "private", "prisons"],
    # 15. Climate Change & Energy
    ["climate", "warming", "emissions", "carbon", "fossil", "green", "energy", "oil",
    "renewable", "solar", "wind", "methane", "epa", "environment", "pollution",
    "sustainability", "gas", "coal", "grid", "electric"],
    # 16. Infrastructure & Transportation
    ["infrastructure", "roads", "bridges", "transit", "rail", "amtrak", "grid", "transportation",
    "highway", "subway", "trains", "tunnel", "construction", "pipes", "broadband",
    "water", "sewage", "logistics", "electric", "vehicles"],
    # 17. Voting Rights & Elections
    ["voting", "voters", "election", "fraud", "ballot", "gerrymandering", "suppression",
    "electoral", "college", "registration", "absentee", "primary", "midterms", "polling", "census",
    "districts", "integrity", "disenfranchisement", "turnout", "early"],
    # 18. Supreme Court & Judiciary
    ["supreme", "court", "scotus", "justices", "judge", "ruling", "bench", "nominee",
    "jurisdiction", "unconstitutional", "precedent", "confirmation", "docket", "circuit", "legal",
    "opinion", "constitution", "judicial", "overturn", "appeal"],
    # 19. Campaign Finance & Corruption
    ["campaign", "finance", "citizens", "united", "pac", "lobbying", "donors", "corruption",
    "superpac", "bribery", "ethics", "lobbyist", "funding", "special", "interests",
    "dark", "money", "disclosure", "clout", "quid"],
    # 20. Executive Power & Investigations
    ["impeachment", "subpoena", "treason", "investigation", "committee", "hearing", "testimony",
    "executive", "privilege", "pardon", "cabinet", "veto", "prosecutor", "contempt", "oversight",
    "scandal", "witness", "perjury", "doj", "order"],
    # 21. Immigration & Border Security
   ["border", "immigration", "immigrants", "wall", "asylum", "ice", "deportation", "migrants",
    "citizenship", "visa", "illegal", "legal", "daca", "patrol", "crossing",
    "refugee", "customs", "undocumented", "naturalization", "h1b"],
    # 22. Russia & Ukraine Conflict
    ["ukraine", "russia", "putin", "zelensky", "nato", "war", "sanctions", "invasion",
    "kremlin", "kyiv", "moscow", "missiles", "intelligence", "offensive", "tanks",
    "aid", "weapons", "ceasefire", "territory", "victory"],
    # 23. China & Global Trade
    ["china", "taiwan", "tariffs", "ccp", "trade", "beijing", "espionage", "manufacturing",
    "semiconductors", "supply", "chain", "exports", "imports", "economy", "sanctions",
    "tensions", "dominance", "competition", "shipping", "xi"],
    # 24. Middle East Relations
    ["israel", "palestine", "gaza", "hamas", "middle", "east", "jewish", "antisemitism",
    "conflict", "iran", "jerusalem", "tel", "aviv", "settlements", "apartheid",
    "hezbollah", "oil", "syria", "treaty", "hostages"],
    # 25. Big Tech, Privacy & Social Media
    ["tech", "privacy", "tiktok", "facebook", "censorship", "algorithm", "data", "monopoly",
    "silicon", "valley", "encryption", "surveillance", "platform", "twitter", "google",
    "antitrust", "regulation", "section", "230", "internet"]
]

In [ ]:

docs = combined_data['Combined_Content'].tolist()

umap_model = UMAP( n_neighbors=15, n_components=5, min_dist=0.0,metric='cosine',random_state=42)

cluster_model = KMeans(n_clusters=49, random_state=42)

topic_model = BERTopic(hdbscan_model=cluster_model,umap_model=umap_model,vectorizer_model=vectorizer_model,seed_topic_list=seed_topic_list,calculate_probabilities=True,verbose=True)
topics, probs = topic_model.fit_transform(docs)
freq = topic_model.get_topic_info()

2026-03-10 23:53:07,383 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1183 [00:00<?, ?it/s]

2026-03-11 00:11:24,550 - BERTopic - Embedding - Completed ✓
2026-03-11 00:11:24,552 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-11 00:11:25,274 - BERTopic - Guided - Completed ✓
2026-03-11 00:11:25,275 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 00:12:40,699 - BERTopic - Dimensionality - Completed ✓
2026-03-11 00:12:40,701 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 00:12:41,067 - BERTopic - Cluster - Completed ✓
2026-03-11 00:12:41,080 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 00:12:44,483 - BERTopic - Representation - Completed ✓


In [ ]:

topic_info = topic_model.get_topic_info()
formatted_labels = []

for index, row in topic_info.iterrows():
    topic_id = row['Topic']
    words = row['Representation']

    if topic_id == -1:
        clean_label = "Other / Irrelevant / Noise"
    else:
        clean_label = ", ".join(words)
    formatted_labels.append(clean_label)
for i, label in enumerate(formatted_labels):
    print(f"Index {i}: {label}")

llm_topic_list = formatted_labels

Index 0: talking, theyre, exactly, real, funny, mean, pretty, god, okay, way
Index 1: senate, house, mcconnell, reagan, theyre, power, white, law, democratic, run
Index 2: capitalism, socialism, social, socialist, china, communist, market, communism, democratic, democracy
Index 3: lie, congress, lying, resign, lies, returns, fraud, oath, jail, hearing
Index 4: ted, lieu, hearing, water, buttery, movie, way, males, dick, dude
Index 5: ukraine, russia, putin, russian, war, nato, invasion, sanctions, military, conflict
Index 6: mueller, barr, letter, barrs, report, summary, investigation, special, muellers, public
Index 7: mueller, letter, public, doj, report, investigation, department, muellers, released, mueller report
Index 8: candidate, primary, warren, poll, candidates, voters, nominee, progressive, polls, pete
Index 9: doj, perjury, congress, testify, contempt, indicted, justice, indict, sitting, mueller
Index 10: report, summary, context, title, headline, word, lie, pages, applied,

In [ ]:
info = topic_model.get_topic_info()
print(info)

    Topic  Count                                           Name  \
0       0   2471                  0_talking_theyre_exactly_real   
1       1   2116                1_senate_house_mcconnell_reagan   
2       2   1876        2_capitalism_socialism_social_socialist   
3       3   1772                    3_lie_congress_lying_resign   
4       4   1716                       4_ted_lieu_hearing_water   
5       5   1670                 5_ukraine_russia_putin_russian   
6       6   1504                    6_mueller_barr_letter_barrs   
7       7   1315                    7_mueller_letter_public_doj   
8       8   1251                8_candidate_primary_warren_poll   
9       9   1220                 9_doj_perjury_congress_testify   
10     10   1180                10_report_summary_context_title   
11     11   1179              11_democracy_theyre_god_education   
12     12   1174          12_voters_electoral_elections_primary   
13     13   1138            13_barr_barrs_credibility_testimon

In [ ]:
topics_to_merge = [
    [38, 43, 44, 46, 47 48],
    [6,7,13],
    [8, 16],
    [25, 42, 42, 45]
]
topic_model.merge_topics(docs, topics_to_merge)

In [ ]:
formatted_labels = []

for topic_id in topic_model.get_topics():

    words_with_scores = topic_model.get_topic(topic_id)
    words_only = [word for word, score in words_with_scores[:20]]
    if topic_id == -1:
        clean_label = "Other / Irrelevant / Noise"
    else:
        clean_label = ", ".join(words_only)
    formatted_labels.append(clean_label)
for i, label in enumerate(formatted_labels):
    print(f"Index {i}: {label}")

Index 0: mueller, barr, report, letter, investigation, doj, congress, public, summary, special
Index 1: voters, candidate, primary, warren, candidates, poll, win, polls, democratic, campaign
Index 2: removed, participating, thank, questions, message, following, question, mistakes, reasons, reason
Index 3: senate, house, mcconnell, reagan, theyre, power, white, law, democratic, way
Index 4: capitalism, socialism, social, socialist, china, market, communist, democratic, communism, democracy
Index 5: congress, lie, lying, resign, lies, returns, fraud, oath, jail, hearing
Index 6: ukraine, russia, putin, russian, war, nato, invasion, sanctions, military, conflict
Index 7: democracy, theyre, god, education, history, far, way, shapiro, different, war
Index 8: nazis, israel, fascism, nazi, leftist, fascist, social, war, jewish, freedom
Index 9: workers, labor, unions, strike, wage, union, minimum, wages, pay, work
Index 10: black, white, race, racism, racist, rights, slavery, police, history,

In [ ]:
topic_mapping = {
    0: "INVESTIGATION: Government Investigations",
    1: "ELECTIONS: Campaigns & Primaries",
    2: "MODERATION: Community Interaction Messages",
    3: "POLITICS: Legislative Branch & Governance",
    4: "IDEOLOGY: Economic & Political Systems",
    5: "POLITICS: Ethical Misconduct & Accountability",
    6: "FOREIGN POLICY: Russia-Ukraine Conflict",
    7: "CULTURE: Values & Historical Identity",
    8: "IDEOLOGY: Extremism & Global Conflict",
    9: "LABOR: Workers Rights & Unions",
    10: "SOCIAL: Race & Civil Rights",
    11: "ECONOMY: Economics & Markets",
    12: "SOCIAL: Gender & Identity",
    13: "MEDIA: News Outlets & Propaganda",
    14: "ENVIRONMENT & IMMIGRATION: Climate & Borders",
    15: "POLITICS: Impeachment & Presidential Trials",
    16: "TECH: Social Media & Platform Rules",
    17: "ECONOMY: Wealth & Taxation",
    18: "POLITICS: Campaign Finance & Corruption",
    19: "HEALTH: Healthcare Policy & Insurance",
    20: "SOCIAL: Reproductive Rights & Abortion",
    21: "FOREIGN POLICY: Latin America & Global Policy",
    22: "POLICY: Violations & Bans",
    23: "HEALTH: COVID-19 & Public Health",
    24: "JUSTICE: Criminal Justice & Incarceration",
    25: "JUSTICE: Gun Control & 2nd Amendment",
    26: "JUSTICE: Policing & Law Enforcement",
    27: "JUSTICE: Supreme Court & Legal Rulings",
    28: "ECONOMY: Housing & Real Estate",
    29: "POLITICS: State Governance & Education",
    30: "SOCIAL: Student Debt & Higher Education",
    31: "JUSTICE: Drug Policy & Legalization",
    32: "NOISE: General Conversational Filler"
}

In [ ]:
import pickle
save_path = '/content/drive/MyDrive/K-means_bertopic_model_data.pkl'
import pickle
with open(save_path, 'wb') as f:
    pickle.dump({
        'vectorizer': vectorizer_model,
        'topic_model': topic_model,
        'human_labels': topic_mapping
    }, f)

print("BERTopic model and labels saved as 'K-means_bertopic_model_data.pkl'")

BERTopic model and labels saved as 'K-means_bertopic_model_data.pkl'
